In [4]:
# Install dependencies
!pip install beautifulsoup4 requests anthropic lxml --break-system-packages

# Run demo with mock data
!python demo.py

# Run with real data
!python run_tracker.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.5/405.5 kB 9.5 MB/s eta 0:00:00
python3: can't open file '/content/demo.py': [Errno 2] No such file or directory
python3: can't open file '/content/run_tracker.py': [Errno 2] No such file or directory


In [6]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import json
import re
from collections import defaultdict
from typing import List, Dict, Tuple
import time


In [25]:
class StockNewsSentimentTracker:
    def __init__(self, companies: List[str], api_key: str = None):
        """
        Initialize the sentiment tracker

        Args:
            companies: List of company stock symbols (e.g., ['AAPL', 'GOOGL', 'MSFT'])
            api_key: Optional API key for Claude API (for sentiment analysis)
        """
        self.companies = companies
        self.api_key = api_key
        self.articles = defaultdict(list)
        self.sentiment_results = {}

    def crawl_yahoo_finance(self, symbol: str, max_articles: int = 10) -> List[Dict]:
        """
        Crawl Yahoo Finance for news articles

        Args:
            symbol: Stock symbol (e.g., 'AAPL')
            max_articles: Maximum number of articles to fetch

        Returns:
            List of article dictionaries
        """
        articles = []
        url = f"https://finance.yahoo.com/quote/{symbol}/news"

        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Debug: Print a snippet of the soup to check if content is fetched
            print(f"Yahoo Finance soup snippet for {symbol}: {soup.prettify()[:2000]}")

            # Yahoo Finance news items - CURRENTLY NOT WORKING, NEEDS DEBUGGING
            news_items = soup.find_all('div', class_=re.compile('Ov\\(h\\)|news-stream'))
            print(f"  - Yahoo Finance: Found {len(news_items)} potential news items before filtering")
            news_items = news_items[:max_articles]

            for item in news_items:
                try:
                    headline_tag = item.find('h3') or item.find('a')
                    if headline_tag:
                        headline = headline_tag.get_text(strip=True)
                        link = headline_tag.get('href', '')

                        if link and not link.startswith('http'):
                            link = f"https://finance.yahoo.com{link}"

                        # Try to get article snippet/summary
                        summary_tag = item.find('p')
                        summary = summary_tag.get_text(strip=True) if summary_tag else ""

                        articles.append({
                            'source': 'Yahoo Finance',
                            'headline': headline,
                            'summary': summary,
                            'link': link,
                            'symbol': symbol,
                            'timestamp': datetime.now().isoformat()
                        })
                except Exception as e:
                    continue

        except Exception as e:
            print(f"Error crawling Yahoo Finance for {symbol}: {e}")

        return articles

    def crawl_reuters(self, symbol: str, company_name: str, max_articles: int = 10) -> List[Dict]:
        """
        Crawl Reuters for news articles

        Args:
            symbol: Stock symbol
            company_name: Full company name for search
            max_articles: Maximum number of articles

        Returns:
            List of article dictionaries
        """
        articles = []
        # Reuters search URL
        search_query = company_name.replace(' ', '+')
        url = f"https://www.reuters.com/site-search/?query={search_query}"

        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Debug: Print a snippet of the soup to check if content is fetched
            print(f"Reuters soup snippet for {symbol}: {soup.prettify()[:2000]}")

            # Find article links and headlines - CURRENTLY NOT WORKING, NEEDS DEBUGGING
            articles_found = soup.find_all('a', attrs={'data-testid': re.compile('Heading|Title')})
            print(f"  - Reuters: Found {len(articles_found)} potential news items before filtering")
            articles_found = articles_found[:max_articles]

            for article in articles_found:
                try:
                    headline = article.get_text(strip=True)
                    link = article.get('href', '')

                    if link and not link.startswith('http'):
                        link = f"https://www.reuters.com{link}"

                    articles.append({
                        'source': 'Reuters',
                        'headline': headline,
                        'summary': '',
                        'link': link,
                        'symbol': symbol,
                        'timestamp': datetime.now().isoformat()
                    })
                except Exception as e:
                    continue

        except Exception as e:
            print(f"Error crawling Reuters for {symbol}: {e}")

        return articles

    def crawl_all_sources(self, company_map: Dict[str, str] = None):
        """
        Crawl all news sources for all companies

        Args:
            company_map: Dictionary mapping symbols to company names
                        e.g., {'AAPL': 'Apple Inc', 'GOOGL': 'Google'}
        """
        if company_map is None:
            company_map = {symbol: symbol for symbol in self.companies}

        print("🔍 Crawling financial news sources...")

        for symbol in self.companies:
            print(f"\n📰 Fetching news for {symbol}...")

            # Yahoo Finance
            yahoo_articles = self.crawl_yahoo_finance(symbol, max_articles=5)
            self.articles[symbol].extend(yahoo_articles)
            print(f"  - Yahoo Finance: {len(yahoo_articles)} articles")

            time.sleep(1)  # Rate limiting

            # Reuters
            if symbol in company_map:
                reuters_articles = self.crawl_reuters(symbol, company_map[symbol], max_articles=5)
                self.articles[symbol].extend(reuters_articles)
                print(f"  - Reuters: {len(reuters_articles)} articles")

            time.sleep(1)  # Rate limiting

        print(f"\n✅ Total articles collected: {sum(len(arts) for arts in self.articles.values())}")

    def analyze_sentiment_local(self, text: str) -> Tuple[str, float, List[str]]:
        """
        Perform basic sentiment analysis using keyword matching
        (Fallback when Claude API is not available)

        Args:
            text: Text to analyze

        Returns:
            Tuple of (sentiment, score, keywords)
        """
        text_lower = text.lower()

        # Positive keywords
        positive_words = [
            'surge', 'soar', 'gain', 'profit', 'growth', 'beat', 'exceed', 'strong',
            'positive', 'bullish', 'upgrade', 'buy', 'success', 'record', 'high',
            'rally', 'rise', 'jump', 'boost', 'expansion', 'innovative', 'outperform'
        ]

        # Negative keywords
        negative_words = [
            'fall', 'drop', 'loss', 'decline', 'miss', 'weak', 'negative', 'bearish',
            'downgrade', 'sell', 'failure', 'low', 'plunge', 'crash', 'concern',
            'risk', 'lawsuit', 'investigation', 'scandal', 'layoff', 'cut', 'underperform'
        ]

        # Count occurrences
        pos_count = sum(1 for word in positive_words if word in text_lower)
        neg_count = sum(1 for word in negative_words if word in text_lower)

        # Find keywords present
        keywords = []
        for word in positive_words + negative_words:
            if word in text_lower:
                keywords.append(word)

        # Determine sentiment
        if pos_count > neg_count:
            sentiment = 'positive'
            score = min(0.5 + (pos_count - neg_count) * 0.1, 1.0)
        elif neg_count > pos_count:
            sentiment = 'negative'
            score = max(-0.5 - (neg_count - pos_count) * 0.1, -1.0)
        else:
            sentiment = 'neutral'
            score = 0.0

        return sentiment, score, keywords[:5]  # Top 5 keywords

    def analyze_sentiment_with_claude(self, text: str, symbol: str) -> Dict:
        """
        Perform sentiment analysis using Claude API

        Args:
            text: Text to analyze
            symbol: Stock symbol

        Returns:
            Dictionary with sentiment analysis results
        """
        if not self.api_key:
            # Fallback to local analysis
            sentiment, score, keywords = self.analyze_sentiment_local(text)
            return {
                'sentiment': sentiment,
                'score': score,
                'keywords': keywords,
                'explanation': f"Based on keyword analysis"
            }

        try:
            prompt = f"""Analyze the sentiment of this financial news for {symbol}:

\""{text}\""

Provide your analysis in the following JSON format:
{{
    "sentiment": "positive|negative|neutral",
    "score": <number between -1 and 1>,
    "keywords": [<list of key terms driving sentiment>],
    "explanation": "<brief explanation of the sentiment>"
}}"""

            response = requests.post(
                "https://api.anthropic.com/v1/messages",
                headers={
                    "Content-Type": "application/json",
                    "x-api-key": self.api_key,
                    "anthropic-version": "2023-06-01"
                },
                json={
                    "model": "claude-sonnet-4-20250514",
                    "max_tokens": 500,
                    "messages": [
                        {"role": "user", "content": prompt}
                    ]
                },
                timeout=30
            )

            if response.status_code == 200:
                result = response.json()
                content = result['content'][0]['text']

                # Extract JSON from response
                json_match = re.search(r'\{.*\}', content, re.DOTALL)
                if json_match:
                    return json.loads(json_match.group())

        except Exception as e:
            print(f"Error with Claude API: {e}")

        # Fallback
        sentiment, score, keywords = self.analyze_sentiment_local(text)
        return {
            'sentiment': sentiment,
            'score': score,
            'keywords': keywords,
            'explanation': 'Based on keyword analysis'
        }

    def analyze_all_articles(self):
        """
        Analyze sentiment for all collected articles
        """
        print("\n🤖 Analyzing sentiment...")

        for symbol, articles in self.articles.items():
            print(f"\n📊 Analyzing {symbol} ({len(articles)} articles)...")

            sentiments = []
            all_keywords = []

            for article in articles:
                text = f"{article['headline']} {article['summary']}"

                # Analyze sentiment
                analysis = self.analyze_sentiment_with_claude(text, symbol)

                article['sentiment'] = analysis['sentiment']
                article['sentiment_score'] = analysis['score']
                article['keywords'] = analysis['keywords']

                sentiments.append(analysis['score'])
                all_keywords.extend(analysis['keywords'])

            # Calculate average sentiment
            if sentiments:
                avg_sentiment = sum(sentiments) / len(sentiments)

                if avg_sentiment > 0.2:
                    overall = 'positive'
                elif avg_sentiment < -0.2:
                    overall = 'negative'
                else:
                    overall = 'neutral'

                # Count keyword frequency
                keyword_freq = defaultdict(int)
                for kw in all_keywords:
                    keyword_freq[kw] += 1

                top_keywords = sorted(keyword_freq.items(), key=lambda x: x[1], reverse=True)[:5]

                self.sentiment_results[symbol] = {
                    'overall_sentiment': overall,
                    'average_score': round(avg_sentiment, 3),
                    'article_count': len(articles),
                    'top_keywords': [kw[0] for kw in top_keywords],
                    'articles': articles
                }

    def generate_insights(self) -> Dict[str, str]:
        """
        Generate human-readable insights for each stock

        Returns:
            Dictionary mapping symbols to insight strings
        """
        insights = {}

        print("\n💡 Generating insights...")

        for symbol, data in self.sentiment_results.items():
            sentiment = data['overall_sentiment']
            score = data['average_score']
            keywords = data['top_keywords']
            article_count = data['article_count']

            # Determine trend
            if sentiment == 'positive':
                trend = "turned positive" if score > 0.5 else "remains cautiously positive"
            elif sentiment == 'negative':
                trend = "turned negative" if score < -0.5 else "shows bearish sentiment"
            else:
                trend = "remains neutral"

            # Build insight
            insight_parts = [
                f"Stock {symbol} sentiment {trend} based on {article_count} recent articles."
            ]

            if keywords:
                keyword_str = ", ".join(keywords[:3])
                insight_parts.append(f"Key drivers: {keyword_str}.")

            # Add specific context from articles
            recent_headlines = [a['headline'] for a in data['articles'][:2]]
            if recent_headlines:
                insight_parts.append(f"Recent news includes: '{recent_headlines[0]}'")

            insights[symbol] = " ".join(insight_parts)

        return insights

    def generate_report(self, output_file: str = None):
        """
        Generate a comprehensive sentiment report

        Args:
            output_file: Optional file path to save report
        """
        insights = self.generate_insights()

        report = {
            'generated_at': datetime.now().isoformat(),
            'summary': {
                symbol: {
                    'sentiment': data['overall_sentiment'],
                    'score': data['average_score'],
                    'article_count': data['article_count'],
                    'top_keywords': data['top_keywords'],
                    'insight': insights.get(symbol, '')
                }
                for symbol, data in self.sentiment_results.items()
            },
            'detailed_articles': self.sentiment_results
        }

        if output_file:
            with open(output_file, 'w') as f:
                json.dump(report, f, indent=2)
            print(f"\n📄 Report saved to: {output_file}")

        return report

    def print_summary(self):
        """
        Print a formatted summary to console
        """
        print("\n" + "="*70)
        print("📈 DAILY STOCK SENTIMENT SUMMARY")
        print("="*70)
        print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

        for symbol in sorted(self.sentiment_results.keys()):
            data = self.sentiment_results[symbol]
            sentiment = data['overall_sentiment']
            score = data['average_score']

            # Color coding for terminal
            if sentiment == 'positive':
                emoji = "📗"
                indicator = "+"
            elif sentiment == 'negative':
                emoji = "📕"
                indicator = "-"
            else:
                emoji = "📘"
                indicator = "○"

            print(f"{emoji} {symbol}")
            print(f"   Sentiment: {sentiment.upper()} ({indicator}{abs(score):.2f})")
            print(f"   Articles: {data['article_count']}")
            print(f"   Keywords: {', '.join(data['top_keywords'][:3])}")

            # Generate and print insight
            insights = self.generate_insights()
            if symbol in insights:
                print(f"   💡 {insights[symbol]}")
            print()


In [26]:
def crawl_reuters(self, symbol: str, company_name: str, max_articles: int = 10) -> List[Dict]:
        """
        Crawl Reuters for news articles

        Args:
            symbol: Stock symbol
            company_name: Full company name for search
            max_articles: Maximum number of articles

        Returns:
            List of article dictionaries
        """
        articles = []
        # Reuters search URL
        search_query = company_name.replace(' ', '+')
        url = f"https://www.reuters.com/site-search/?query={search_query}"

        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Find article links and headlines
            articles_found = soup.find_all('a', attrs={'data-testid': re.compile('Heading|Title')})[:max_articles]

            for article in articles_found:
                try:
                    headline = article.get_text(strip=True)
                    link = article.get('href', '')

                    if link and not link.startswith('http'):
                        link = f"https://www.reuters.com{link}"

                    articles.append({
                        'source': 'Reuters',
                        'headline': headline,
                        'summary': '',
                        'link': link,
                        'symbol': symbol,
                        'timestamp': datetime.now().isoformat()
                    })
                except Exception as e:
                    continue

        except Exception as e:
            print(f"Error crawling Reuters for {symbol}: {e}")

        return articles

In [9]:
def crawl_all_sources(self, company_map: Dict[str, str] = None):
        """
        Crawl all news sources for all companies

        Args:
            company_map: Dictionary mapping symbols to company names
                        e.g., {'AAPL': 'Apple Inc', 'GOOGL': 'Google'}
        """
        if company_map is None:
            company_map = {symbol: symbol for symbol in self.companies}

        print("🔍 Crawling financial news sources...")

        for symbol in self.companies:
            print(f"\n📰 Fetching news for {symbol}...")

            # Yahoo Finance
            yahoo_articles = self.crawl_yahoo_finance(symbol, max_articles=5)
            self.articles[symbol].extend(yahoo_articles)
            print(f"  - Yahoo Finance: {len(yahoo_articles)} articles")

            time.sleep(1)  # Rate limiting

            # Reuters
            if symbol in company_map:
                reuters_articles = self.crawl_reuters(symbol, company_map[symbol], max_articles=5)
                self.articles[symbol].extend(reuters_articles)
                print(f"  - Reuters: {len(reuters_articles)} articles")

            time.sleep(1)  # Rate limiting

        print(f"\n✅ Total articles collected: {sum(len(arts) for arts in self.articles.values())}")


In [10]:
def analyze_sentiment_local(self, text: str) -> Tuple[str, float, List[str]]:
        """
        Perform basic sentiment analysis using keyword matching
        (Fallback when Claude API is not available)

        Args:
            text: Text to analyze

        Returns:
            Tuple of (sentiment, score, keywords)
        """
        text_lower = text.lower()

        # Positive keywords
        positive_words = [
            'surge', 'soar', 'gain', 'profit', 'growth', 'beat', 'exceed', 'strong',
            'positive', 'bullish', 'upgrade', 'buy', 'success', 'record', 'high',
            'rally', 'rise', 'jump', 'boost', 'expansion', 'innovative', 'outperform'
        ]

        # Negative keywords
        negative_words = [
            'fall', 'drop', 'loss', 'decline', 'miss', 'weak', 'negative', 'bearish',
            'downgrade', 'sell', 'failure', 'low', 'plunge', 'crash', 'concern',
            'risk', 'lawsuit', 'investigation', 'scandal', 'layoff', 'cut', 'underperform'
        ]

        # Count occurrences
        pos_count = sum(1 for word in positive_words if word in text_lower)
        neg_count = sum(1 for word in negative_words if word in text_lower)

        # Find keywords present
        keywords = []
        for word in positive_words + negative_words:
            if word in text_lower:
                keywords.append(word)

        # Determine sentiment
        if pos_count > neg_count:
            sentiment = 'positive'
            score = min(0.5 + (pos_count - neg_count) * 0.1, 1.0)
        elif neg_count > pos_count:
            sentiment = 'negative'
            score = max(-0.5 - (neg_count - pos_count) * 0.1, -1.0)
        else:
            sentiment = 'neutral'
            score = 0.0

        return sentiment, score, keywords[:5]  # Top 5 keywords

In [11]:
def analyze_sentiment_with_claude(self, text: str, symbol: str) -> Dict:
        """
        Perform sentiment analysis using Claude API

        Args:
            text: Text to analyze
            symbol: Stock symbol

        Returns:
            Dictionary with sentiment analysis results
        """
        if not self.api_key:
            # Fallback to local analysis
            sentiment, score, keywords = self.analyze_sentiment_local(text)
            return {
                'sentiment': sentiment,
                'score': score,
                'keywords': keywords,
                'explanation': f"Based on keyword analysis"
            }

        try:
            prompt = f"""Analyze the sentiment of this financial news for {symbol}:

"{text}"

Provide your analysis in the following JSON format:
{{
    "sentiment": "positive|negative|neutral",
    "score": <number between -1 and 1>,
    "keywords": [<list of key terms driving sentiment>],
    "explanation": "<brief explanation of the sentiment>"
}}"""

            response = requests.post(
                "https://api.anthropic.com/v1/messages",
                headers={
                    "Content-Type": "application/json",
                    "x-api-key": self.api_key,
                    "anthropic-version": "2023-06-01"
                },
                json={
                    "model": "claude-sonnet-4-20250514",
                    "max_tokens": 500,
                    "messages": [
                        {"role": "user", "content": prompt}
                    ]
                },
                timeout=30
            )

            if response.status_code == 200:
                result = response.json()
                content = result['content'][0]['text']

                # Extract JSON from response
                json_match = re.search(r'\{.*\}', content, re.DOTALL)
                if json_match:
                    return json.loads(json_match.group())

        except Exception as e:
            print(f"Error with Claude API: {e}")

        # Fallback
        sentiment, score, keywords = self.analyze_sentiment_local(text)
        return {
            'sentiment': sentiment,
            'score': score,
            'keywords': keywords,
            'explanation': 'Based on keyword analysis'
        }

In [12]:
def analyze_all_articles(self):
        """Analyze sentiment for all collected articles"""
        print("\n🤖 Analyzing sentiment...")

        for symbol, articles in self.articles.items():
            print(f"\n📊 Analyzing {symbol} ({len(articles)} articles)...")

            sentiments = []
            all_keywords = []

            for article in articles:
                text = f"{article['headline']} {article['summary']}"

                # Analyze sentiment
                analysis = self.analyze_sentiment_with_claude(text, symbol)

                article['sentiment'] = analysis['sentiment']
                article['sentiment_score'] = analysis['score']
                article['keywords'] = analysis['keywords']

                sentiments.append(analysis['score'])
                all_keywords.extend(analysis['keywords'])

            # Calculate average sentiment
            if sentiments:
                avg_sentiment = sum(sentiments) / len(sentiments)

                if avg_sentiment > 0.2:
                    overall = 'positive'
                elif avg_sentiment < -0.2:
                    overall = 'negative'
                else:
                    overall = 'neutral'

                # Count keyword frequency
                keyword_freq = defaultdict(int)
                for kw in all_keywords:
                    keyword_freq[kw] += 1

                top_keywords = sorted(keyword_freq.items(), key=lambda x: x[1], reverse=True)[:5]

                self.sentiment_results[symbol] = {
                    'overall_sentiment': overall,
                    'average_score': round(avg_sentiment, 3),
                    'article_count': len(articles),
                    'top_keywords': [kw[0] for kw in top_keywords],
                    'articles': articles
                }

In [13]:
def generate_insights(self) -> Dict[str, str]:
        """
        Generate human-readable insights for each stock

        Returns:
            Dictionary mapping symbols to insight strings
        """
        insights = {}

        print("\n💡 Generating insights...")

        for symbol, data in self.sentiment_results.items():
            sentiment = data['overall_sentiment']
            score = data['average_score']
            keywords = data['top_keywords']
            article_count = data['article_count']

            # Determine trend
            if sentiment == 'positive':
                trend = "turned positive" if score > 0.5 else "remains cautiously positive"
            elif sentiment == 'negative':
                trend = "turned negative" if score < -0.5 else "shows bearish sentiment"
            else:
                trend = "remains neutral"

            # Build insight
            insight_parts = [
                f"Stock {symbol} sentiment {trend} based on {article_count} recent articles."
            ]

            if keywords:
                keyword_str = ", ".join(keywords[:3])
                insight_parts.append(f"Key drivers: {keyword_str}.")

            # Add specific context from articles
            recent_headlines = [a['headline'] for a in data['articles'][:2]]
            if recent_headlines:
                insight_parts.append(f"Recent news includes: '{recent_headlines[0]}'")

            insights[symbol] = " ".join(insight_parts)

        return insights

In [16]:
    def generate_report(self, output_file: str = None):
        """
        Generate a comprehensive sentiment report

        Args:
            output_file: Optional file path to save report
        """
        insights = self.generate_insights()

        report = {
            'generated_at': datetime.now().isoformat(),
            'summary': {
                symbol: {
                    'sentiment': data['overall_sentiment'],
                    'score': data['average_score'],
                    'article_count': data['article_count'],
                    'top_keywords': data['top_keywords'],
                    'insight': insights.get(symbol, '')
                }
                for symbol, data in self.sentiment_results.items()
            },
            'detailed_articles': self.sentiment_results
        }

        if output_file:
            with open(output_file, 'w') as f:
                json.dump(report, f, indent=2)
            print(f"\n📄 Report saved to: {output_file}")

        return report

    def print_summary(self):
        """Print a formatted summary to console"""
        print("\n" + "="*70)
        print("📈 DAILY STOCK SENTIMENT SUMMARY")
        print("="*70)
        print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

        for symbol in sorted(self.sentiment_results.keys()):
            data = self.sentiment_results[symbol]
            sentiment = data['overall_sentiment']
            score = data['average_score']

            # Color coding for terminal
            if sentiment == 'positive':
                emoji = "📗"
                indicator = "+"
            elif sentiment == 'negative':
                emoji = "📕"
                indicator = "-"
            else:
                emoji = "📘"
                indicator = "○"

            print(f"{emoji} {symbol}")
            print(f"   Sentiment: {sentiment.upper()} ({indicator}{abs(score):.2f})")
            print(f"   Articles: {data['article_count']}")
            print(f"   Keywords: {', '.join(data['top_keywords'][:3])}")

            # Generate and print insight
            insights = self.generate_insights()
            if symbol in insights:
                print(f"   💡 {insights[symbol]}")
            print()


In [22]:
def main():
    """Example usage"""
    # Define companies to track
    companies = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'NVDA']

    # Map to full company names for better search results
    company_map = {
        'AAPL': 'Apple Inc',
        'GOOGL': 'Google Alphabet',
        'MSFT': 'Microsoft',
        'TSLA': 'Tesla',
        'NVDA': 'NVIDIA'
    }

    # Initialize tracker
    # Note: Add your Claude API key here for AI-powered analysis
    # tracker = StockNewsSentimentTracker(companies, api_key="your-api-key-here")
    tracker = StockNewsSentimentTracker(companies)

    # Crawl news sources
    tracker.crawl_all_sources(company_map)

    # Analyze sentiment
    tracker.analyze_all_articles()

    # Print summary
    tracker.print_summary()

    # Generate full report
    report = tracker.generate_report('/content/sentiment_report.json')

    print("\n✅ Analysis complete!")


if __name__ == "__main__":
    main()

🔍 Crawling financial news sources...

📰 Fetching news for AAPL...
  - Yahoo Finance: Found 0 potential news items before filtering
  - Yahoo Finance: 0 articles
  - Reuters: Found 0 potential news items before filtering
  - Reuters: 0 articles

📰 Fetching news for GOOGL...
  - Yahoo Finance: Found 0 potential news items before filtering
  - Yahoo Finance: 0 articles
  - Reuters: Found 0 potential news items before filtering
  - Reuters: 0 articles

📰 Fetching news for MSFT...
  - Yahoo Finance: Found 0 potential news items before filtering
  - Yahoo Finance: 0 articles
  - Reuters: Found 0 potential news items before filtering
  - Reuters: 0 articles

📰 Fetching news for TSLA...
  - Yahoo Finance: Found 0 potential news items before filtering
  - Yahoo Finance: 0 articles
  - Reuters: Found 0 potential news items before filtering
  - Reuters: 0 articles

📰 Fetching news for NVDA...
  - Yahoo Finance: Found 0 potential news items before filtering
  - Yahoo Finance: 0 articles
  - Reuter